In [1]:
rows = []

In [16]:
import json
from datetime import datetime
import pandas as pd

# -----------------------------
# CONFIG
# -----------------------------
JSON_FILE_PATH = "..\\data\\wayfair_response_production_FROM_2024-10-29 00_21_06.000000 +00_00_LIMIT_50_ORDER_BY_ASC.json"
# START_DATE = datetime(2025, 12, 13)
# END_DATE = datetime(2025, 12, 15)

# -----------------------------
# LOAD JSON
# -----------------------------
with open(JSON_FILE_PATH, "r") as f:
    raw_json = json.load(f)

orders = raw_json["data"]["getDropshipPurchaseOrders"]



# -----------------------------
# PARSE & MAP DATA
# -----------------------------
for order in orders:
    po_date_str = order.get("poDate")

    if not po_date_str:
        continue

    # Convert poDate to datetime
    po_date = datetime.strptime(
        po_date_str.split(".")[0], "%Y-%m-%d %H:%M:%S"
    )

    # # Filter date range (Dec 13–15)
    # if not (START_DATE <= po_date <= END_DATE):
    #     continue

    customer_postal_code = order.get("customerPostalCode")
    customer_city = order.get("customerCity")

    # Prefer orderType, fallback to salesChannelName
    customer_type = order.get("orderType") 

    for product in order.get("products", []):
        rows.append({
            "Date": po_date.date(),
            "sku_id": product.get("sku"),
            "unit_sold": int(product.get("quantity", 0)),
            "unit_price": product.get("price"),
            "customer_postal_code": customer_postal_code,
            "customer_city": customer_city,
            "customer_type": customer_type
        })

# -----------------------------
# CREATE DATAFRAME
# -----------------------------
# df = pd.DataFrame(rows)

# print(df)

# -----------------------------
# OPTIONAL: SAVE TO CSV
# -----------------------------
# df.to_csv("Braxton_data_jan_2024_dec16_2025.csv", index=False)


In [17]:
len(rows), len(orders)

(5974, 50)

In [18]:
# CREATE DATAFRAME
# -----------------------------
df = pd.DataFrame(rows)

print(df)

# -----------------------------
# OPTIONAL: SAVE TO CSV
# -----------------------------
df.to_csv("Braxton_data_jan_2024_dec16_2025.csv", index=False)


            Date    sku_id  unit_sold  unit_price customer_postal_code  \
0     2024-11-01  BXCM1783          1       367.5           60614-3706   
1     2024-11-01  BXCL1785          2       356.5           53066-1658   
2     2024-11-01  BXCM3934          1       950.5                63073   
3     2024-11-01  BXCM5353          1        15.0                84042   
4     2024-11-01  BXCM5368          1        15.0                84042   
...          ...       ...        ...         ...                  ...   
5969  2024-11-06  BXCM1656          2       241.0           29492-8421   
5970  2024-11-06  BXCM4490          1       417.0           78373-4129   
5971  2024-11-06  BXCM3690          1       587.5           33139-1356   
5972  2024-11-06  BXCM1635          1       697.5           33139-1356   
5973  2024-11-06  BXCM4525          1       268.5                46765   

     customer_city customer_type  
0          Chicago           B2B  
1       Oconomowoc          None  
2     